# RAG — Hands-on from Scratch (LangChain v1)

**Scenario:** A **customer support assistant** for *Blue Reef Aquarium* (a sample company).
The assistant will answer questions based on a maintenance guide, a document on aquatic diseases, a return policy, and a transcript of a customer conversation—**citing its sources** for every answer. If asked about something not present in the knowledge base, it will respond by saying, "I don't know."

Notebook workflow (execute each step sequentially and examine the outputs):

| Step | Action | RAG Component |
|---|---|---|
| 0 | Setup | – |
| 1 | Create a sample knowledge base | Source Data |
| 2 | Load documents | **Load** |
| 3 | Test 6 different chunking methods | **Split** |
| 4 | Embeddings + vector database | **Embed + Store** |
| 5 | Retriever (similarity, MMR, filtering) | **Retrieve** |
| 6 | Generate answers using an LLM (with citations) | **Generate** |
| 7 | Compare chunking strategies via **evaluation** | **Evaluate** |
| 8 | Hybrid search (BM25 + vector) | Advanced |
| 9 | Build and compare the same assistant using **CAG** | Alternative Architecture |

## 0) Setup

For embeddings, we use a **local and free** multilingual model (`intfloat/multilingual-e5-base`);
For generation, we use Claude.

In [1]:
import os,textwrap, statistics
from pathlib import Path

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"   

import warnings
warnings.filterwarnings("ignore")

!pip install -q -U langchain-core langchain-text-splitters langchain-classic langchain-anthropic \
    langchain-huggingface sentence-transformers langchain-experimental langchain-community \
    rank_bm25 tiktoken beautifulsoup4 lxml python-dotenv


In [2]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
    print("The Key Imported From Secrets!")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    print("The Key Imported from .env file !!")

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY doesnt exist !: The chunking/retrieval cells work, but the LLM cells do not.")

LLM_MODEL = "claude-sonnet-5-5" 

def show(chunks, n=None, max_chars=None):
    """It prints the chunks and their basic statistics—we will use this when comparing each method."""
    lengths = [len(c.page_content) for c in chunks]
    print(f"Totally {len(chunks)} chunk | mean {statistics.mean(lengths):.0f} characters | "
          f"min {min(lengths)} | max {max(lengths)}\n")

    for i, c in enumerate(chunks[:n] if n else chunks):
        meta = {k: v for k, v in c.metadata.items() if k != "source"}
        print(f"┌── chunk {i} | {len(c.page_content)} kr | {c.metadata.get('source', '')} {meta if meta else ''}")
        text = c.page_content
        if max_chars and len(text) > max_chars:   
            text = text[:max_chars] + " …"
        for line in text.splitlines():
            print(f"│ {line}")
        print("└" + "─" * 70 + "\n")

The Key Imported From Secrets!


## 1) Sample knowledge base

We are intentionally preparing four documents with **distinct structures**, as the optimal chunking strategy depends on the structure of the document:

- `bakim_rehberi.md` → contains headings and tables (**structured** document)
- `hastaliklar.md` → contains headings and short sections
- `iade_politikasi.txt` → plain text, organized as a list of points
- `musteri_gorusmesi.txt` → **narrative** text (a transcript) that shifts between various topics

In [3]:
DATA = Path("data"); DATA.mkdir(exist_ok=True)

(DATA / "bakim_rehberi.md").write_text("""# Tatlı Su Akvaryumu Bakım Rehberi

## 1. Su Parametreleri

Balıkların sağlığı için su değerlerinin düzenli ölçülmesi gerekir. Aşağıdaki tablo yaygın türler için ideal aralıkları gösterir.

| Tür | pH | Sıcaklık (°C) | Sertlik (GH) |
|---|---|---|---|
| Neon Tetra | 6.0 - 7.0 | 22 - 26 | 1 - 10 |
| Lepistes (Guppy) | 7.0 - 8.0 | 22 - 28 | 8 - 12 |
| Japon Balığı | 7.0 - 7.5 | 18 - 22 | 5 - 19 |
| Melek Balığı | 6.5 - 7.5 | 24 - 30 | 3 - 8 |

Amonyak ve nitrit her zaman 0 ppm olmalıdır. Nitrat ise 20 ppm'in altında tutulmalıdır.

## 2. Azot Döngüsü ve İlk Kurulum

Yeni kurulan bir akvaryumda faydalı bakteri kolonisi henüz oluşmamıştır. Bu bakteriler balık atıklarından oluşan amonyağı önce nitrite, sonra daha az zararlı olan nitrata dönüştürür. Bu sürece azot döngüsü denir ve genellikle 4 ila 6 hafta sürer. Döngü tamamlanmadan akvaryuma balık eklenmemelidir. Döngüyü hızlandırmak için bakteri kültürü takviyesi kullanılabilir.

## 3. Haftalık Bakım

Her hafta akvaryum suyunun yüzde 20 ile 25'i değiştirilmelidir. Yeni eklenen su, akvaryum suyuyla aynı sıcaklıkta olmalı ve mutlaka su düzenleyici ile klordan arındırılmalıdır. Dip süpürgesi ile kum yüzeyindeki yem artıkları ve dışkılar temizlenmelidir. Cam yüzeylerdeki yosun, yosun kazıyıcı ile alınabilir.

## 4. Filtre Bakımı

Filtre, faydalı bakterilerin en yoğun yaşadığı yerdir. Bu nedenle filtre süngeri asla musluk suyu ile yıkanmamalıdır; musluk suyundaki klor bakteri kolonisini öldürür. Sünger, su değişimi sırasında akvaryumdan alınan suyla bir kova içinde hafifçe sıkılarak temizlenmelidir. Filtre malzemelerinin hepsi aynı anda değiştirilmemeli, değişim kademeli yapılmalıdır.

## 5. Yemleme

Balıklar günde 1 veya 2 kez, 2-3 dakikada tüketebilecekleri kadar beslenmelidir. Fazla yem suyun bozulmasının en yaygın sebebidir. Haftada bir gün yem verilmemesi sindirim sistemi için faydalıdır.
""", encoding="utf-8")

(DATA / "hastaliklar.md").write_text("""# Sık Görülen Balık Hastalıkları

## Beyaz Benek (Ich)

Belirtiler: Balığın vücudunda ve yüzgeçlerinde tuz tanesi büyüklüğünde beyaz noktalar görülür. Balık kendini dekora sürter.
Tedavi: Su sıcaklığı günde 1 derece artırılarak kademeli olarak 28-30°C'ye çıkarılır. Bu, parazitin yaşam döngüsünü hızlandırır. Beyaz benek ilacı paket talimatına göre uygulanır ve tedavi süresince aktif karbon filtreden çıkarılır.

## Yüzgeç Çürümesi

Belirtiler: Yüzgeç kenarları beyazlaşır, yıpranır ve giderek kısalır.
Sebep: Genellikle kötü su kalitesine bağlı bakteriyel enfeksiyondur.
Tedavi: Önce su değerleri düzeltilir, ardından antibakteriyel ilaç kullanılır.

## Kadife Hastalığı

Belirtiler: Balığın üzerinde altın sarısı ya da pas renginde, toz gibi bir tabaka oluşur.
Tedavi: Akvaryum karartılır, çünkü parazit ışıkla fotosentez yapar. Bakır içerikli ilaç uygulanır; omurgasız canlılar varsa bakır kullanılmamalıdır.
""", encoding="utf-8")

(DATA / "iade_politikasi.txt").write_text("""MAVİ RESİF AKVARYUM — İADE VE CANLI GARANTİ POLİTİKASI

Canlı Balık Garantisi
Kargo ile gönderilen canlı balıklarda teslimattan sonraki 48 saat içinde ölüm yaşanırsa, ölü balığın fotoğrafı ile birlikte müşteri hizmetlerine bildirim yapılmalıdır. Bildirim 48 saat içinde yapıldığında aynı tür ücretsiz olarak yeniden gönderilir veya ücret iadesi yapılır. 48 saati geçen bildirimler garanti kapsamına girmez.

Ekipman İadesi
Filtre, ısıtıcı, lamba gibi ekipmanlar kullanılmamış ve orijinal ambalajında olmak şartıyla 14 gün içinde iade edilebilir. Kullanılmış ekipmanlarda yalnızca üretici garantisi geçerlidir.

Kargo Ücreti
1500 TL ve üzeri siparişlerde kargo ücretsizdir. Canlı balık gönderimleri yalnızca Pazartesi, Salı ve Çarşamba günleri yapılır; böylece balıkların hafta sonu kargoda beklemesi önlenir.
""", encoding="utf-8")

(DATA / "musteri_gorusmesi.txt").write_text("""Müşteri: Merhaba, iki gündür akvaryumumun suyu süt gibi bulanık. Ne yapmam gerekiyor? Temsilci: Merhaba, akvaryumunuz yeni mi kuruldu? Müşteri: Evet, on gün önce kurdum. Temsilci: O zaman bu büyük ihtimalle bakteri patlamasıdır. Yeni akvaryumlarda sık görülür ve genellikle birkaç gün içinde kendiliğinden geçer. Bu süreçte yemi azaltın ve kimyasal berraklaştırıcı kullanmayın. Müşteri: Anladım, teşekkürler. Bir de gelecek ay on gün tatile gideceğim. Balıklarımı nasıl besleyeceğim? Temsilci: Sağlıklı yetişkin balıklar birkaç gün aç kalabilir ama on gün uzun bir süre. Size otomatik yemlik öneririm. Tatilden önce bir hafta deneyip doğru miktarı ayarlayın. Tatil yemi denen jel tabletler ise suyu kirletebildiği için tercih etmiyoruz. Müşteri: Peki otomatik yemliği sipariş etsem ne zaman gelir? Temsilci: Ekipman siparişleri aynı gün kargoya verilir ve genellikle iki iş günü içinde teslim edilir. Siparişiniz 1500 TL'yi geçerse kargo ücretsiz olur.
""", encoding="utf-8")

print(sorted(p.name for p in DATA.iterdir()))

['bakim_rehberi.md', 'hastaliklar.md', 'iade_politikasi.txt', 'musteri_gorusmesi.txt']


## 2) Load — Loading documents

In LangChain, everything is a `Document` object: `page_content` (text) + `metadata` (source, page number, title, etc.).

**Metadata is the unsung hero of RAG:** it is essential for citing sources, filtering, and debugging.

In a real-world project, you might use: `PyPDFLoader` or `PyMuPDFLoader` for PDFs; `unstructured` or `docling` for complex PDFs or tables;
`WebBaseLoader` for web content; or a custom loader for your specific database.

In [4]:
from langchain_core.documents import Document

docs = [
    Document(page_content=p.read_text(encoding="utf-8"), metadata={"source": p.name})
    for p in sorted(DATA.iterdir())
]
for d in docs:
    print(f"{d.metadata['source']:<25} {len(d.page_content):>5} character")

bakim_rehberi.md           1865 character
hastaliklar.md              913 character
iade_politikasi.txt         809 character
musteri_gorusmesi.txt       953 character


## 3) Split — Chunking Lab

We will split the same data using different methods and **visually compare the outputs**. In step 7, we will also measure them quantitatively.

### 3a) Naive method: `CharacterTextSplitter` (reference / "bad example")
This method splits text based on a single separator. It might cut through the middle of a sentence or even a table. We include it here for demonstration purposes.

In [5]:
from langchain_text_splitters import CharacterTextSplitter

naive = CharacterTextSplitter(separator=" ", chunk_size=300, chunk_overlap=0)
naive_chunks = naive.split_documents(docs)
show(naive_chunks, n=4)
# 👀 Note: The chunk boundaries, while not occurring in the middle of a word, do fall in the middle of a sentence or section.

Totally 18 chunk | mean 251 characters | min 24 | max 300

┌── chunk 0 | 299 kr | bakim_rehberi.md 
│ # Tatlı Su Akvaryumu Bakım Rehberi
│ 
│ ## 1. Su Parametreleri
│ 
│ Balıkların sağlığı için su değerlerinin düzenli ölçülmesi gerekir. Aşağıdaki tablo yaygın türler için ideal aralıkları gösterir.
│ 
│ | Tür | pH | Sıcaklık (°C) | Sertlik (GH) |
│ |---|---|---|---|
│ | Neon Tetra | 6.0 - 7.0 | 22 - 26 | 1 - 10 |
│ |
└──────────────────────────────────────────────────────────────────────

┌── chunk 1 | 297 kr | bakim_rehberi.md 
│ Lepistes (Guppy) | 7.0 - 8.0 | 22 - 28 | 8 - 12 |
│ | Japon Balığı | 7.0 - 7.5 | 18 - 22 | 5 - 19 |
│ | Melek Balığı | 6.5 - 7.5 | 24 - 30 | 3 - 8 |
│ 
│ Amonyak ve nitrit her zaman 0 ppm olmalıdır. Nitrat ise 20 ppm'in altında tutulmalıdır.
│ 
│ ## 2. Azot Döngüsü ve İlk Kurulum
│ 
│ Yeni kurulan bir akvaryumda
└──────────────────────────────────────────────────────────────────────

┌── chunk 2 | 296 kr | bakim_rehberi.md 
│ faydalı bakteri kolonisi henüz olu

### 3b) Recursive — `RecursiveCharacterTextSplitter` (row 1 in the table)

It attempts a list of separators **in order**: first `\n\n` (paragraph), then `\n` (line), then `. ` (sentence), and finally a space.
The underlying logic is: "preserve the largest meaningful unit possible; if it doesn't fit, move down to the next level."

Two critical parameters:
- **`chunk_size`**: The upper limit (measured in *characters* by default).
- **`chunk_overlap`**: The portion shared between consecutive chunks. This ensures that information split at a boundary remains available with its context on both sides. Typically set to 10–20% of the `chunk_size`.

`add_start_index=True` → records the position of the chunk within the original document in the metadata (useful for source citation or highlighting).

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " ", ""],   # We have also added the end of the sentence for the Turkish text
    keep_separator="end",                       # The "." should be attached to the end of the sentence, not to the beginning of the next chunk
    add_start_index=True,
)
recursive_chunks = recursive.split_documents(docs)
show(recursive_chunks, n=6)

Totally 17 chunk | mean 273 characters | min 54 | max 398

┌── chunk 0 | 188 kr | bakim_rehberi.md {'start_index': 0}
│ # Tatlı Su Akvaryumu Bakım Rehberi
│ 
│ ## 1. Su Parametreleri
│ 
│ Balıkların sağlığı için su değerlerinin düzenli ölçülmesi gerekir. Aşağıdaki tablo yaygın türler için ideal aralıkları gösterir.
└──────────────────────────────────────────────────────────────────────

┌── chunk 1 | 378 kr | bakim_rehberi.md {'start_index': 190}
│ | Tür | pH | Sıcaklık (°C) | Sertlik (GH) |
│ |---|---|---|---|
│ | Neon Tetra | 6.0 - 7.0 | 22 - 26 | 1 - 10 |
│ | Lepistes (Guppy) | 7.0 - 8.0 | 22 - 28 | 8 - 12 |
│ | Japon Balığı | 7.0 - 7.5 | 18 - 22 | 5 - 19 |
│ | Melek Balığı | 6.5 - 7.5 | 24 - 30 | 3 - 8 |
│ 
│ Amonyak ve nitrit her zaman 0 ppm olmalıdır. Nitrat ise 20 ppm'in altında tutulmalıdır.
│ 
│ ## 2. Azot Döngüsü ve İlk Kurulum
└──────────────────────────────────────────────────────────────────────

┌── chunk 2 | 388 kr | bakim_rehberi.md {'start_index': 570}
│ Yeni kurulan b

#### ⚠️ Characters or tokens? Don’t exceed your embedding model’s limit!

Embedding models have a **maximum token** limit (e.g., `multilingual-e5-base` → 512 tokens;
some MiniLM models → 128 tokens). If a chunk exceeds this limit, it is **silently truncated**, rendering the end of the text unsearchable.
Therefore, the best practice in production is to measure chunk size using the model’s own tokenizer:

```python
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tok, chunk_size=256, chunk_overlap=32)
```
Since Turkish is an agglutinative language, the same text will contain **more tokens** than it would in English—be sure to account for this.

In [7]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tok, chunk_size=256, chunk_overlap=32)

### 3c) Layout-Aware — `MarkdownHeaderTextSplitter` (3rd row in the table)

It uses the document's **inherent structure** (headings) as boundaries and records these headings in the **metadata**.
This ensures that a chunk within the "Filter Maintenance" section, for example, "knows" which section it belongs to.

Practical approach: **first split based on structure, then recursively split any remaining sections that are too long** (a two-stage process).

Other layout-aware splitters include: `HTMLHeaderTextSplitter`, `HTMLSemanticPreservingSplitter` (which preserves tables),
`RecursiveJsonSplitter`, and—for code—`RecursiveCharacterTextSplitter.from_language(Language.PYTHON, ...)`.
When working with PDFs, a common method is to first convert the document into Markdown using tools like `docling` or `unstructured`, and then apply this splitter.

In [8]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "doküman"), ("##", "bölüm")],
    strip_headers=False,   # keep the heading text within the chunk as well → it adds meaning to the embedding
)
second_step = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80, separators=["\n\n", "\n", ". ", " "],
                                              keep_separator="end")

layout_chunks = []
for d in docs:
    if d.metadata["source"].endswith(".md"):
        parts = md_splitter.split_text(d.page_content)          # Stage 1: by headings
        for p in parts:
            p.metadata["source"] = d.metadata["source"]
        layout_chunks += second_step.split_documents(parts)    # Step 2: Divide if it's long
    else:
        layout_chunks += second_step.split_documents([d])         # Falling into recursive in unstructured documents

show(layout_chunks)
# 👀 The table has remained intact, and each chunk has a 'section' metadata field.

Totally 12 chunk | mean 378 characters | min 212 | max 570

┌── chunk 0 | 537 kr | bakim_rehberi.md {'doküman': 'Tatlı Su Akvaryumu Bakım Rehberi', 'bölüm': '1. Su Parametreleri'}
│ # Tatlı Su Akvaryumu Bakım Rehberi  
│ ## 1. Su Parametreleri  
│ Balıkların sağlığı için su değerlerinin düzenli ölçülmesi gerekir. Aşağıdaki tablo yaygın türler için ideal aralıkları gösterir.  
│ | Tür | pH | Sıcaklık (°C) | Sertlik (GH) |
│ |---|---|---|---|
│ | Neon Tetra | 6.0 - 7.0 | 22 - 26 | 1 - 10 |
│ | Lepistes (Guppy) | 7.0 - 8.0 | 22 - 28 | 8 - 12 |
│ | Japon Balığı | 7.0 - 7.5 | 18 - 22 | 5 - 19 |
│ | Melek Balığı | 6.5 - 7.5 | 24 - 30 | 3 - 8 |  
│ Amonyak ve nitrit her zaman 0 ppm olmalıdır. Nitrat ise 20 ppm'in altında tutulmalıdır.
└──────────────────────────────────────────────────────────────────────

┌── chunk 1 | 402 kr | bakim_rehberi.md {'doküman': 'Tatlı Su Akvaryumu Bakım Rehberi', 'bölüm': '2. Azot Döngüsü ve İlk Kurulum'}
│ ## 2. Azot Döngüsü ve İlk Kurulum  
│ Yeni kurulan bir a

### 3d) Semantic — split based on breaks in the flow of meaning (row 2 in the table)

**Let’s walk through the logic step-by-step to ensure it’s clear:**
1. Break the text down into individual sentences.
2. Generate an embedding for each sentence (incorporating its neighbors).
3. Calculate the **cosine distance** between consecutive sentences.
4. If the distance exceeds a certain threshold (e.g., the 80th percentile of all calculated distances) → identify this as a "topic shift" and split the text at that point.

This approach excels with texts that lack headings and jump between topics, such as transcripts. The trade-offs: it requires an embedding API call for every sentence (which can be slow and costly) and necessitates the configuration of a threshold value (`breakpoint_threshold_amount`).

In [9]:
import re, numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

# The E5 models were trained using "query: " / "passage: " prefixes; in LangChain, we provide this via `encode_kwargs`.
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    encode_kwargs={"prompt": "passage: ", "normalize_embeddings": True},
    query_encode_kwargs={"prompt": "query: ", "normalize_embeddings": True},
)

def semantic_split(text, embeddings, percentile=80, buffer=1):
    sentences = [c for c in re.split(r"(?<=[.?!])\s+", text.strip()) if c]
    # Embed each sentence together with its preceding/following 'buffer' sentences → reduces the noise of an isolated sentence
    windows = [" ".join(sentences[max(0, i - buffer): i + buffer + 1]) for i in range(len(sentences))]
    vecs = np.array(embeddings.embed_documents(windows))
    ranges = [1 - float(np.dot(vecs[i], vecs[i + 1])) for i in range(len(vecs) - 1)]  # normalize → dot = cos
    threshold = np.percentile(ranges, percentile)
    chunks, curr = [], [sentences[0]]
    for i, m in enumerate(ranges):
        if m > threshold:
            chunks.append(" ".join(curr)); curr = []
        curr.append(sentences[i + 1])
    chunks.append(" ".join(curr))
    return chunks, ranges, threshold

transkript = next(d for d in docs if d.metadata["source"] == "musteri_gorusmesi.txt")
parts, ranges, threshold = semantic_split(transkript.page_content, embeddings, percentile=80)
print(f"Threshold: {threshold:.3f}\n")
for i, p in enumerate(parts):
    print(f"--- semantic chunk {i}\n{p}\n")
# 👀 Expected: 'bulanik su', 'tatil beslemesi', 'kargo/teslimat' topics should be in separate chunks.
# If it does not decrease, move the percentile value between 70-95 and observe the effect.

Threshold: 0.060

--- semantic chunk 0
Müşteri: Merhaba, iki gündür akvaryumumun suyu süt gibi bulanık. Ne yapmam gerekiyor? Temsilci: Merhaba, akvaryumunuz yeni mi kuruldu? Müşteri: Evet, on gün önce kurdum. Temsilci: O zaman bu büyük ihtimalle bakteri patlamasıdır. Yeni akvaryumlarda sık görülür ve genellikle birkaç gün içinde kendiliğinden geçer. Bu süreçte yemi azaltın ve kimyasal berraklaştırıcı kullanmayın.

--- semantic chunk 1
Müşteri: Anladım, teşekkürler.

--- semantic chunk 2
Bir de gelecek ay on gün tatile gideceğim. Balıklarımı nasıl besleyeceğim? Temsilci: Sağlıklı yetişkin balıklar birkaç gün aç kalabilir ama on gün uzun bir süre. Size otomatik yemlik öneririm.

--- semantic chunk 3
Tatilden önce bir hafta deneyip doğru miktarı ayarlayın. Tatil yemi denen jel tabletler ise suyu kirletebildiği için tercih etmiyoruz. Müşteri: Peki otomatik yemliği sipariş etsem ne zaman gelir? Temsilci: Ekipman siparişleri aynı gün kargoya verilir ve genellikle iki iş günü içinde teslim ed

The equivalent of this in LangChain is `SemanticChunker` (found within `langchain_experimental`).

⚠️ The `langchain-experimental` package is no longer under active development (it is being sunset); while it is fine for learning purposes, for production use, you should either write your own function—as shown above—or use a pinned version of the package.

In [10]:
from langchain_experimental.text_splitter import SemanticChunker

semantic = SemanticChunker(embeddings, breakpoint_threshold_type="percentile", breakpoint_threshold_amount=80)
semantic_chunks = [c for c in semantic.split_documents(docs) if c.page_content.strip()]  
show(semantic_chunks)

Totally 14 chunk | mean 323 characters | min 30 | max 732

┌── chunk 0 | 126 kr | bakim_rehberi.md 
│ # Tatlı Su Akvaryumu Bakım Rehberi
│ 
│ ## 1. Su Parametreleri
│ 
│ Balıkların sağlığı için su değerlerinin düzenli ölçülmesi gerekir.
└──────────────────────────────────────────────────────────────────────

┌── chunk 1 | 405 kr | bakim_rehberi.md 
│ Aşağıdaki tablo yaygın türler için ideal aralıkları gösterir. | Tür | pH | Sıcaklık (°C) | Sertlik (GH) |
│ |---|---|---|---|
│ | Neon Tetra | 6.0 - 7.0 | 22 - 26 | 1 - 10 |
│ | Lepistes (Guppy) | 7.0 - 8.0 | 22 - 28 | 8 - 12 |
│ | Japon Balığı | 7.0 - 7.5 | 18 - 22 | 5 - 19 |
│ | Melek Balığı | 6.5 - 7.5 | 24 - 30 | 3 - 8 |
│ 
│ Amonyak ve nitrit her zaman 0 ppm olmalıdır. Nitrat ise 20 ppm'in altında tutulmalıdır.
└──────────────────────────────────────────────────────────────────────

┌── chunk 2 | 732 kr | bakim_rehberi.md 
│ ## 2. Azot Döngüsü ve İlk Kurulum
│ 
│ Yeni kurulan bir akvaryumda faydalı bakteri kolonisi henüz oluşmamıştır.

### 3e) Agentic — Splitting via LLM

We provide the text to an LLM and instruct it to "break the text into segments that are self-contained and express a single idea."
A common variant is **proposition chunking**, where the LLM ensures each segment is independent of the surrounding context
(e.g., "This process takes 4–6 weeks" → "The nitrogen cycle takes 4–6 weeks"). This resolves issues with pronouns—the quality is high,
but since it requires an LLM call for every document, it is **slow and expensive**. It is a sensible approach for a small number of highly valuable documents.

In [11]:
from pydantic import BaseModel, Field
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model=LLM_MODEL)

class Part(BaseModel):
    header: str = Field(description="A short title describing the subject of the piece")
    content: str = Field(description="A text segment that is understandable on its own; pronouns have been replaced with explicit nouns.")

class Parsing(BaseModel):
    parts: list[Part]

agentic_prompt = """Aşağıdaki metni bir RAG sistemi için parçalara böl.
Kurallar:
- Her parça TEK bir konu/fikir içersin ve diğer parçalar okunmadan anlaşılabilsin.
- 'bu', 'o', 'bunlar' gibi bağlama muhtaç ifadeleri açık isimlerle değiştir.
- Bilgi ekleme veya çıkarma; sadece yeniden düzenle.

METİN:
{text}"""

def agentic_split(doc: Document) -> list[Document]:
    result = llm.with_structured_output(Parsing).invoke(agentic_prompt.format(text=doc.page_content))
    return [Document(page_content=f"{p.header}\n{p.content}",
                     metadata={**doc.metadata, "header": p.header}) for p in result.parts]

guide_pages = next(d for d in docs if d.metadata["source"] == "bakim_rehberi.md")
agentic_chunks = agentic_split(guide_pages)
show(agentic_chunks)

Totally 10 chunk | mean 211 characters | min 97 | max 451

┌── chunk 0 | 451 kr | bakim_rehberi.md {'header': 'Su Parametrelerinin Ölçülmesi ve Türlere Göre İdeal Aralıklar'}
│ Su Parametrelerinin Ölçülmesi ve Türlere Göre İdeal Aralıklar
│ Tatlı su akvaryumunda balıkların sağlığı için su değerlerinin düzenli ölçülmesi gerekir. Yaygın türler için ideal aralıklar şöyledir:
│ 
│ | Tür | pH | Sıcaklık (°C) | Sertlik (GH) |
│ |---|---|---|---|
│ | Neon Tetra | 6.0 - 7.0 | 22 - 26 | 1 - 10 |
│ | Lepistes (Guppy) | 7.0 - 8.0 | 22 - 28 | 8 - 12 |
│ | Japon Balığı | 7.0 - 7.5 | 18 - 22 | 5 - 19 |
│ | Melek Balığı | 6.5 - 7.5 | 24 - 30 | 3 - 8 |
└──────────────────────────────────────────────────────────────────────

┌── chunk 1 | 145 kr | bakim_rehberi.md {'header': 'Amonyak, Nitrit ve Nitrat Sınırları'}
│ Amonyak, Nitrit ve Nitrat Sınırları
│ Tatlı su akvaryumunda amonyak ve nitrit her zaman 0 ppm olmalıdır. Nitrat ise 20 ppm'in altında tutulmalıdır.
└─────────────────────────────────────────

### 3f) Small-to-Big — `ParentDocumentRetriever`

The fundamental dilemma of chunking:
- **Small chunk** → the embedding represents a single idea → **precise search**, but the context provided to the LLM is poor.
- **Large chunk** → rich in context, but the embedding becomes an average of many ideas → **search becomes imprecise (blurry)**.

The "Small-to-Big" approach decouples these: **search using small chunks, but provide the larger chunk (the "parent" to which the matching small chunk belongs) to the LLM.**
- Vector DB → stores small "child" chunks (which are embedded).
- Docstore → stores large "parent" chunks (which are not embedded; they are retrieved via an ID).

Since this method operates during the retrieval phase, it requires the vector DB mentioned in step 4; we will define it here and utilize it in step 5.

## 4) Embed + Store — Vector Database

Each chunk is converted into a vector and stored. For learning purposes, `InMemoryVectorStore` is sufficient; for production, tools like **Chroma**, **Qdrant**, or **pgvector** (pgvector is highly practical if you are already using PostgreSQL) are used.

In [12]:
from langchain_core.vectorstores import InMemoryVectorStore

vs_layout = InMemoryVectorStore(embeddings)
vs_layout.add_documents(layout_chunks)

v = embeddings.embed_query("filtre nasıl temizlenir")
print(f"Vector Size: {len(v)}  | top-5 value: {np.round(v[:5], 4)}")

Vector Size: 768  | top-5 value: [-0.0017  0.0428 -0.0132  0.0513  0.0153]


Setup for Small-to-Big: parent = the size of a section, child = the size of 1–2 sentences.

In [13]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=0, separators=["\n## ", "\n\n", "\n", ". "])
child_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20, separators=["\n", ". ", " "], keep_separator="end")

small2big = ParentDocumentRetriever(
    vectorstore=InMemoryVectorStore(embeddings),   # children are here (they are embedded)
    docstore=InMemoryStore(),                      # parents are here (by ID)
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 3},
)
small2big.add_documents(docs)

prompt = "Filtre süngerini musluk suyuyla yıkayabilir miyim?"
print("CHILD matches (the smaller segments being searched for):")
for c in small2big.vectorstore.similarity_search(prompt, k=5):
    print("  •", c.page_content[:120])
print("\nPARENT results (large chunks to be sent to the LLM):")
for p in small2big.invoke(prompt):
    print(f"  • [{p.metadata['source']}] {len(p.page_content)} kr → {p.page_content[:120]}…")

CHILD matches (the smaller segments being searched for):
  • Bu nedenle filtre süngeri asla musluk suyu ile yıkanmamalıdır; musluk suyundaki klor bakteri kolonisini öldürür.
  • Sünger, su değişimi sırasında akvaryumdan alınan suyla bir kova içinde hafifçe sıkılarak temizlenmelidir.
  • Filtre malzemelerinin hepsi aynı anda değiştirilmemeli, değişim kademeli yapılmalıdır.
  • Filtre, ısıtıcı, lamba gibi ekipmanlar kullanılmamış ve orijinal ambalajında olmak şartıyla 14 gün içinde iade edilebili
  • Dip süpürgesi ile kum yüzeyindeki yem artıkları ve dışkılar temizlenmelidir. Cam yüzeylerdeki yosun, yosun kazıyıcı ile 

PARENT results (large chunks to be sent to the LLM):
  • [bakim_rehberi.md] 594 kr → ## 4. Filtre Bakımı

Filtre, faydalı bakterilerin en yoğun yaşadığı yerdir. Bu nedenle filtre süngeri asla musluk suyu i…


## 5) Retrieve — Configure the Retriever

The Retriever acts as an interface that "takes a query and returns relevant Documents." Key settings include:
- **`k`**: The number of chunks to return. Too few → risk of missing information; too many → excess noise + higher costs. A good starting point is 3–5.
- **`search_type="mmr"`** (Maximal Marginal Relevance): Ensures the returned results are **diverse**, rather than containing redundant or duplicate chunks.
- **Metadata filtering**: E.g., searching only within the "return policy" document. (In production environments, filtering by department, date, or authorization level is crucial.)

In [14]:
retriever = vs_layout.as_retriever(search_type="similarity", search_kwargs={"k": 4})

for d, score in vs_layout.similarity_search_with_score("Neon tetra için pH kaç olmalı?", k=3):
    print(f"{score:.3f} | {d.metadata.get('bölüm', d.metadata['source'])}")

# MMR: fetch_k aday arasından, birbirine benzemeyen k tanesini seçer
mmr = vs_layout.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.5})

# Metadata filtresi (InMemoryVectorStore'da filtre bir fonksiyondur; Chroma/pgvector'da dict sözdizimi kullanılır)
sadece_iade = vs_layout.as_retriever(
    search_kwargs={"k": 2, "filter": lambda d: d.metadata["source"] == "iade_politikasi.txt"})


print([d.metadata["source"] for d in sadece_iade.invoke("kargo ne zaman gelir")])

0.852 | 1. Su Parametreleri
0.789 | 2. Azot Döngüsü ve İlk Kurulum
0.785 | Beyaz Benek (Ich)
['iade_politikasi.txt', 'iade_politikasi.txt']


## 6) Generate — Produce a response based on sources

The prompt has 3 functions:
1. **Grounding**: "Use only the provided context" → reduces hallucinations.
2. **Escape hatch**: "If it’s not in the context, state that you don’t know" → prevents fabrication.
3. **Citing sources**: Allows both you and the user to verify the answer.

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

SYSTEM = """Sen Mavi Resif Akvaryum'un müşteri destek asistanısın.
Soruları YALNIZCA <baglam> içindeki bilgilere dayanarak cevapla.
Cevap bağlamda yoksa "Bu konuda bilgi bankamda bilgi bulamadım, müşteri hizmetlerimize yönlendirebilirim." de; asla tahmin yürütme.
Kullandığın her bilginin sonuna kaynağını [Kaynak N] şeklinde ekle. Kısa ve net cevap ver."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "<baglam>\n{context}\n</baglam>\n\nSoru: {question}"),
])

def format_docs(docs):
    return "\n\n".join(
        f"[Kaynak {i+1}] ({d.metadata['source']}{' › ' + d.metadata['bölüm'] if 'bölüm' in d.metadata else ''})\n{d.page_content}"
        for i, d in enumerate(docs)
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

for q in ["Filtre süngerini musluk suyuyla yıkayabilir miyim?",
          "Beyaz benek tedavisinde sıcaklığı kaç dereceye çıkarmalıyım?",
          "Deniz akvaryumu için hangi tuzu önerirsiniz?"]:          # ← bilgi bankasında YOK
    print(f"❓ {q}\n💬 {rag_chain.invoke(q)}\n")

❓ Filtre süngerini musluk suyuyla yıkayabilir miyim?
💬 Hayır, filtre süngerini musluk suyuyla yıkamamalısınız. Musluk suyundaki klor, filtredeki faydalı bakteri kolonisini öldürür. Süngeri, su değişimi sırasında akvaryumdan aldığınız suyla bir kova içinde hafifçe sıkarak temizleyin. [Kaynak 1]

❓ Beyaz benek tedavisinde sıcaklığı kaç dereceye çıkarmalıyım?
💬 Su sıcaklığını günde 1 derece artırarak kademeli şekilde 28-30°C'ye çıkarmalısınız. Bu, parazitin yaşam döngüsünü hızlandırır. [Kaynak 1]

Ayrıca beyaz benek ilacını paket talimatına göre uygulayın ve tedavi süresince aktif karbonu filtreden çıkarın. [Kaynak 1]

❓ Deniz akvaryumu için hangi tuzu önerirsiniz?
💬 Bu konuda bilgi bankamda bilgi bulamadım, müşteri hizmetlerimize yönlendirebilirim.



**Debugging habit:** If the answer is poor, look at *retrieval* first, then *generation*.
Most poor answers stem from the retrieval of incorrect or incomplete chunks, not from the LLM itself.

In [16]:
q = "Canlı balık ölürse kaç saat içinde bildirmeliyim?"
for d in retriever.invoke(q):
    print(f"• {d.metadata['source']} | {d.page_content[:100]}…")

• iade_politikasi.txt | MAVİ RESİF AKVARYUM — İADE VE CANLI GARANTİ POLİTİKASI

Canlı Balık Garantisi
Kargo ile gönderilen c…
• musteri_gorusmesi.txt | Müşteri: Merhaba, iki gündür akvaryumumun suyu süt gibi bulanık. Ne yapmam gerekiyor? Temsilci: Merh…
• iade_politikasi.txt | Ekipman İadesi
Filtre, ısıtıcı, lamba gibi ekipmanlar kullanılmamış ve orijinal ambalajında olmak şa…
• bakim_rehberi.md | ## 2. Azot Döngüsü ve İlk Kurulum  
Yeni kurulan bir akvaryumda faydalı bakteri kolonisi henüz oluşm…


## 7) Evaluate — Compare chunking strategies by MEASURING them

The only honest answer to the question, "Which chunking strategy is better?" is: **measure it using your own data.**
We set up a simple retrieval test: for each question, there is a key phrase that must appear within the correct answer.
**Hit@k** = Is the chunk containing the correct information among the top *k* results?

In a production environment, you would perform this using 50–200 real user queries; for evaluating generation quality, tools like **RAGAS** (which measures faithfulness, answer relevancy, and context precision/recall) are used.

In [17]:
test_seti = [
    ("Filtre süngerini musluk suyuyla yıkayabilir miyim?",           "klor bakteri kolonisini"),
    ("Beyaz benek tedavisinde sıcaklık kaç olmalı?",                 "28-30°C"),
    ("Kargoda ölen balık için ne kadar sürem var?",                  "48 saat içinde"),
    ("Neon tetra hangi pH aralığını sever?",                        "6.0 - 7.0"),
    ("Tatile gidince balıkları nasıl beslerim?",                     "otomatik yemlik"),
    ("Yeni akvaryuma ne zaman balık koyabilirim?",                   "4 ila 6 hafta"),
    ("Kadife hastalığında ışığı açık mı tutmalıyım?",                "karartılır"),
    ("Akvaryumum yeni kuruldu ve su bulanık, neden?",                "bakteri patlaması"),
    ("Hangi günler canlı balık gönderiyorsunuz?",                    "Pazartesi, Salı ve Çarşamba"),
    ("Kullanılmış ısıtıcıyı iade edebilir miyim?",                   "üretici garantisi"),
]

def hit_at_k(retriever, k_label=""):
    isabet = 0
    for soru, anahtar in test_seti:
        sonuc = retriever.invoke(soru)
        bulundu = any(anahtar in d.page_content for d in sonuc)
        isabet += bulundu
        if not bulundu:
            print(f"   ✗ kaçırdı: {soru}")
    return isabet / len(test_seti)

def vektor_retriever(chunks, k=3):
    vs = InMemoryVectorStore(embeddings); vs.add_documents(chunks)
    return vs.as_retriever(search_kwargs={"k": k})

stratejiler = {
    "Naive (300, overlap 0)": vektor_retriever(naive_chunks),
    "Recursive (400/60)":     vektor_retriever(recursive_chunks),
    "Layout-aware (MD+rec)":  vektor_retriever(layout_chunks),
    "Semantic (p80)":         vektor_retriever(semantic_chunks),
    "Small-to-Big":           small2big,
}
sonuclar = {}
for ad, r in stratejiler.items():
    print(f"▶ {ad}")
    sonuclar[ad] = hit_at_k(r)
print()
for ad, skor in sorted(sonuclar.items(), key=lambda x: -x[1]):
    print(f"{ad:<25} Hit@3 = {skor:.0%}  {'█' * int(skor * 20)}")

▶ Naive (300, overlap 0)
▶ Recursive (400/60)
▶ Layout-aware (MD+rec)
   ✗ kaçırdı: Tatile gidince balıkları nasıl beslerim?
▶ Semantic (p80)
▶ Small-to-Big

Naive (300, overlap 0)    Hit@3 = 100%  ████████████████████
Recursive (400/60)        Hit@3 = 100%  ████████████████████
Semantic (p80)            Hit@3 = 100%  ████████████████████
Small-to-Big              Hit@3 = 100%  ████████████████████
Layout-aware (MD+rec)     Hit@3 = 90%  ██████████████████


🧪 **Experiments (do it yourself):**
1. In the recursive method, set `chunk_size` to 150, 400, and 1000 — why does it miss information when the size is small, and why does it also fail when the size is large?
2. Set `chunk_overlap=0` — is information lost when it happens to be split at the boundary?
3. Re-evaluate using `k=1` and `k=5`.
4. In the semantic method, set the percentile to 70 and 95.

Differences might be subtle in a small dataset; they become much more apparent in complex, multi-hundred-page corporate documents with varied formatting.

## 8) Advanced Level: Hybrid Search (BM25 + Vector)

Vector search captures **meaning**, but it can be weak when it comes to **exact matches**—such as product codes, order numbers, specific price values ​​(e.g., "1500 TL"), or proper nouns.
BM25 (classic keyword search) excels in these scenarios. Combining the two—typically via Reciprocal Rank Fusion (RRF)—is the industry standard for production systems.

Further layers of optimization (worth exploring in this order):
**Reranker** (re-rank the top 20 results using a cross-encoder, such as `BAAI/bge-reranker-v2-m3` or Cohere Rerank) →
**Query rewriting / Multi-query** (generate multiple variations of the query using an LLM) → **Contextual retrieval** (prepend 1–2 sentences to each chunk—generated by an LLM—that describe its specific location or context within the original document).

In [18]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

bm25 = BM25Retriever.from_documents(layout_chunks); bm25.k = 3
hybrid = EnsembleRetriever(retrievers=[bm25, vektor_retriever(layout_chunks)], weights=[0.4, 0.6])

print("Hybrid Hit@3:", f"{hit_at_k(hybrid):.0%}")
for d in hybrid.invoke("1500 TL kargo"):
    print("•", d.metadata["source"], "|", d.page_content[:90])

Hybrid Hit@3: 100%
• musteri_gorusmesi.txt | Size otomatik yemlik öneririm. Tatilden önce bir hafta deneyip doğru miktarı ayarlayın. Ta
• iade_politikasi.txt | Ekipman İadesi
Filtre, ısıtıcı, lamba gibi ekipmanlar kullanılmamış ve orijinal ambalajınd
• iade_politikasi.txt | MAVİ RESİF AKVARYUM — İADE VE CANLI GARANTİ POLİTİKASI

Canlı Balık Garantisi
Kargo ile gö
• musteri_gorusmesi.txt | Müşteri: Merhaba, iki gündür akvaryumumun suyu süt gibi bulanık. Ne yapmam gerekiyor? Tems


## 9) Alternative architecture: CAG (Cache-Augmented Generation)

In CAG, there is **no** retrieval: the **entire** knowledge base is included in the prompt, and this static portion is stored in a **prompt cache**.
For subsequent queries, the model does not need to re-process the same context; tokens read from the cache are significantly cheaper and faster to process.

In Anthropic’s API, you can implement this using `cache_control`. (Note: To be eligible for caching, the context must exceed a minimum length—approximately 1024+ tokens—as defined by the model; a small sample dataset might fall below this threshold.
When you test it with actual documents, you will see the `cache_read` value being utilized.)

In [19]:
from langchain_core.messages import SystemMessage, HumanMessage

tum_bilgi = "\n\n".join(f"<dokuman ad='{d.metadata['source']}'>\n{d.page_content}\n</dokuman>" for d in docs)

cag_system = SystemMessage(content=[{
    "type": "text",
    "text": SYSTEM.replace("<baglam>", "<bilgi_bankasi>") + f"\n\n<bilgi_bankasi>\n{tum_bilgi}\n</bilgi_bankasi>",
    "cache_control": {"type": "ephemeral"},     # ← bu blok önbelleğe alınır
}])

def cag_sor(soru):
    yanit = llm.invoke([cag_system, HumanMessage(content=soru)])
    detay = yanit.usage_metadata.get("input_token_details", {})
    print(f"❓ {soru}\n💬 {yanit.content}\n   📊 input={yanit.usage_metadata['input_tokens']} | "
          f"cache_yazma={detay.get('cache_creation') or 0} | cache_okuma={detay.get('cache_read') or 0}\n")

cag_sor("Kargoda ölen balık için ne kadar sürem var?")
cag_sor("Tatile gidince balıkları nasıl beslerim?")   # ikinci çağrıda cache_okuma dolu olmalı

❓ Kargoda ölen balık için ne kadar sürem var?
💬 [{'signature': 'CAQSvwcKEAgUGAI4AUIIdGhpbmtpbmcSDEuF4JDeQ6jDF/jORhoMeTcp7en66k3ZZMjrIjATJhQr6G+FmUpjuYwjRZtyYWvlw8z5DuYsb/wTaQ7EFjF7iALiC+1Pws23iYCkGdEq3AY0tE2EMYQzqoVkUq+96B/WOzyaCGu5BQGCz/Ul3EExBAajC6myySvvWbVDESzP0gXnyB6+MMwUawnF2m4iq/wLw86DiIUDPSy5Nxfxi8QpYuwJCYFV3BZUFRxG+7ETqcJU1kWcpZahWkvT+Sgf7RPSxAqYy/g5BnNF7ScOU4lLOFafaASzDYS1Anp/px38cyxasgORvL9aQDvcyueZNeEiLT9tZ9JP8BTEQr1PrfdrPg3hlGWPXicFva+luCufovK+A4m5ZHqe/DR/XndB+oAVZmVv5SoWIQBa+BbUQGVtNpQPVBIDbJQ3PMf4BjmQplaAf5FqCsYQgVCs1JW3wsgKWfz2EnBWQczwXWuYvJq3sk/eBVPOOqTgHhHVZM2+tP9O7gSv7kZV5Lf4uzuacPgOA+FPi29sgSOWDfGYHmyVoP4ktkf5i1IQkkT5TRHHmNzvympzz11Ih619cuYwR03mv0eECMpIBSqXYK6cGBC0jydBJFKcrwERpIEq7i+nP5ZhnK5xpIr/cD3DXb3MPfEPDZGt1LPPyZ4ruQn3OiScByfZI3qx8LDo2JcaNfL3kmW0KLRp81JyZjkMv2JRddyj2Ysg6Mw3heJsQXNXNBsZa0G8dr5sMIVPKKSlDbnibHE+uIk+1WFXb+4RPuLtPK0KfqSk4hbDh10amdgdzWfYrYC3PNt7Uft1eGBmQL7496Nov30mkPfzCvP3DOq/IP0g0HFVosEf/Tvd7zbObRm9gMYJ3jGimslF8NzUoDDFHw6pO1GtHopTC9ZgddoHDb8NDIsUDLYt

### RAG vs. CAG — what you’ve seen in this notebook

| | RAG (steps 2–8) | CAG (step 9) |
|---|---|---|
| Setup | Loader, splitter, embedding, vector DB, retriever | Prompt only |
| Chunking concerns | Yes (a key determinant of success) | None |
| Risk of retrieval errors | Yes (incorrect chunk → incorrect answer) | None — the model sees everything |
| Scale | Millions of documents | Limited by the context window |
| Cost per query | Low (only for *k* chunks) | Proportional to context size (reduced via caching) |
| Updates | Re-index the modified document | Refresh the cache |
| Access control (ACL) | Personalized via metadata filtering | Difficult (everyone sees the same context) |

**Next step:** The components in this notebook can evolve into *agentic RAG* using LangGraph:
"Does the question require the knowledge base? → search → are the retrieved chunks sufficient? → if not, rewrite the question and search again."
In other words, this serves as a bridge toward AI workflows.